In [1]:
# Google Colab setup: fetch this repository and use this notebook's directory.
from pathlib import Path
import os

REPO_ROOT = Path('/content/BITS_programming')
if not REPO_ROOT.exists():
    !git clone https://github.com/aqwertyuiop48/BITS_programming.git /content/BITS_programming

NOTEBOOK_DIR = REPO_ROOT / 'module_2/week_6/use_case_2'
os.chdir(NOTEBOOK_DIR)
print(f'Working directory: {NOTEBOOK_DIR}')

Cloning into '/content/BITS_programming'...
remote: Enumerating objects: 2101, done.
remote: Counting objects: 100% (35/35), done.
remote: Compressing objects: 100% (30/30), done.
remote: Total 2101 (delta 11), reused 15 (delta 3), pack-reused 2066 (from 1)
Receiving objects: 100% (2101/2101), 263.06 MiB | 21.77 MiB/s, done.
Resolving deltas: 100% (400/400), done.
Updating files: 100% (1348/1348), done.
Working directory: /content/BITS_programming/module_2/week_6/use_case_2


In [2]:
# AWS credentials from Google Colab Secrets
# Makes boto3 / PySpark / AWS access work inside Colab.
import os

def get_colab_secret(name, required=True):
    try:
        from google.colab import userdata
        value = userdata.get(name)
    except Exception as exc:
        if required:
            raise RuntimeError(f"Unable to read Colab Secret: {name}") from exc
        return None
    if required and (value is None or value == ""):
        raise RuntimeError(f"Add the Colab Secret {name} and grant this notebook access.")
    return value

AWS_ACCESS_KEY_ID = get_colab_secret("AWS_ACCESS_KEY_ID")
AWS_SECRET_ACCESS_KEY = get_colab_secret("AWS_SECRET_ACCESS_KEY")
AWS_SESSION_TOKEN = get_colab_secret("AWS_SESSION_TOKEN", required=False)

os.environ["AWS_ACCESS_KEY_ID"] = AWS_ACCESS_KEY_ID
os.environ["AWS_SECRET_ACCESS_KEY"] = AWS_SECRET_ACCESS_KEY
if AWS_SESSION_TOKEN:
    os.environ["AWS_SESSION_TOKEN"] = AWS_SESSION_TOKEN

# Region: change this if your S3 buckets / Glue jobs live in another region.
_aws_region = os.getenv("AWS_REGION") or os.getenv("AWS_DEFAULT_REGION") or "ap-south-1"
os.environ["AWS_REGION"] = _aws_region
os.environ["AWS_DEFAULT_REGION"] = _aws_region

print(f"AWS credentials loaded. Region: {_aws_region}")

AWS credentials loaded. Region: ap-south-1


In [3]:
!pip install boto3
import os
import boto3
from botocore.exceptions import ClientError
from google.colab import userdata

# 1. Inject credentials from Colab Secrets into os.environ
os.environ["AWS_ACCESS_KEY_ID"] = userdata.get("AWS_ACCESS_KEY_ID")
os.environ["AWS_SECRET_ACCESS_KEY"] = userdata.get("AWS_SECRET_ACCESS_KEY")
try:
    os.environ["AWS_SESSION_TOKEN"] = userdata.get("AWS_SESSION_TOKEN")
except Exception:
    pass  # Session token is optional for non-temporary IAM credentials

# 2. Resolve Region
region = os.environ.get("AWS_REGION", "ap-south-1")

# 3. Get AWS Account ID to build globally unique bucket names
sts_client = boto3.client("sts", region_name=region)
account_id = sts_client.get_caller_identity()["Account"]

_s3 = boto3.client("s3", region_name=region)

# 4. Append account_id to guarantee unique names
BASE_BUCKET_NAMES = ['usecase-etl-1', 'usecase-etl-2']
REQUIRED_BUCKETS = [f"{b}-{account_id}" for b in BASE_BUCKET_NAMES]

CREATED_BUCKETS = []

# 5. Create buckets cleanly
for _b in REQUIRED_BUCKETS:
    try:
        if region == "us-east-1":
            _s3.create_bucket(Bucket=_b)
        else:
            _s3.create_bucket(
                Bucket=_b,
                CreateBucketConfiguration={"LocationConstraint": region}
            )
        CREATED_BUCKETS.append(_b)
        print(f"Created bucket: {_b}")
    except ClientError as _e:
        _code = _e.response.get("Error", {}).get("Code", "")
        if _code in ("BucketAlreadyExists", "BucketAlreadyOwnedByYou", "Conflict"):
            print(f"Bucket already exists (reusing existing): {_b}")
        else:
            print(f"Could not create bucket {_b} ({_code}): {_e}")

print("Buckets ready:", REQUIRED_BUCKETS)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.0/140.0 kB 9.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.8/15.8 MB 60.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.2/90.2 kB 5.1 MB/s eta 0:00:00
Bucket already exists (reusing existing): usecase-etl-1-455865672536
Created bucket: usecase-etl-2-455865672536
Buckets ready: ['usecase-etl-1-455865672536', 'usecase-etl-2-455865672536']


# Notebook 3 — Simple ETL Pipeline

**Use Case 2 goal:** demonstrate the ETL pattern end to end — **Extract → Transform → Aggregate → Load**.

## What learners should understand
- how the cleaned file becomes an ETL input
- how aggregation turns row-level transactions into reporting-ready outputs
- how the same business logic can later be recreated in AWS Glue with PySpark


## Dependencies, AWS setup, and files used

### Python packages
- **boto3** — AWS-friendly access to the cleaned file in S3 and the final ETL outputs written back to S3
- **pandas** — easiest way to explain ETL aggregation logic cell by cell
- **io** — bridges S3 object content and pandas DataFrames
- **pathlib** — fallback for offline rehearsal


In [4]:
import os
from pathlib import Path
from io import BytesIO, StringIO
import boto3
import pandas as pd
from google.colab import userdata

# 1. Load AWS credentials from Google Colab Secrets
os.environ["AWS_ACCESS_KEY_ID"] = userdata.get("AWS_ACCESS_KEY_ID")
os.environ["AWS_SECRET_ACCESS_KEY"] = userdata.get("AWS_SECRET_ACCESS_KEY")
try:
    os.environ["AWS_SESSION_TOKEN"] = userdata.get("AWS_SESSION_TOKEN")
except Exception:
    pass

# 2. Set AWS Region and retrieve AWS Account ID dynamically
AWS_REGION = os.environ.get("AWS_REGION", "ap-south-1")

sts_client = boto3.client("sts", region_name=AWS_REGION)
account_id = sts_client.get_caller_identity()["Account"]

s3_client = boto3.client("s3", region_name=AWS_REGION)

# 3. Dynamic bucket names matching Notebook 2 naming convention
BUCKET_1 = f"usecase-etl-1-{account_id}"
BUCKET_2 = f"usecase-etl-2-{account_id}"

# 4. Dynamic S3 URIs
INPUT_S3_URI = f"s3://{BUCKET_1}/processed/retail_cleaned.csv"
OUT_DAILY_S3_URI = f"s3://{BUCKET_2}/output/daily_country_revenue.csv"
OUT_MONTHLY_S3_URI = f"s3://{BUCKET_2}/output/monthly_category_revenue.csv"

# Local fallbacks
LOCAL_INPUT_PATH = Path('./retail_cleaned.csv')
LOCAL_DAILY_PATH = Path('./daily_country_revenue.csv')
LOCAL_MONTHLY_PATH = Path('./monthly_category_revenue.csv')


def parse_s3_uri(uri: str):
    bucket, key = uri.replace('s3://', '', 1).split('/', 1)
    return bucket, key


def read_csv_aws_first(s3_uri: str, local_path: Path) -> pd.DataFrame:
    try:
        bucket, key = parse_s3_uri(s3_uri)
        obj = s3_client.get_object(Bucket=bucket, Key=key)
        print("✅ Reading from S3:", s3_uri)
        return pd.read_csv(BytesIO(obj['Body'].read()))
    except Exception as e:
        print(f"⚠️ S3 failed: {e}")
        print("📂 Falling back to local:", local_path)
        return pd.read_csv(local_path)


def write_csv_aws_first(df: pd.DataFrame, s3_uri: str, local_path: Path) -> None:
    csv_buffer = StringIO()
    df.to_csv(csv_buffer, index=False)

    try:
        bucket, key = parse_s3_uri(s3_uri)
        s3_client.put_object(Bucket=bucket, Key=key, Body=csv_buffer.getvalue().encode('utf-8'))
        print("✅ Written to S3:", s3_uri)
    except Exception as e:
        print(f"⚠️ S3 write failed: {e}")
        print("📂 Writing locally:", local_path)
        local_path.parent.mkdir(parents=True, exist_ok=True)
        local_path.write_text(csv_buffer.getvalue(), encoding='utf-8')


print('ETL input:', INPUT_S3_URI)
print('Daily output:', OUT_DAILY_S3_URI)
print('Monthly output:', OUT_MONTHLY_S3_URI)

ETL input: s3://usecase-etl-1-455865672536/processed/retail_cleaned.csv
Daily output: s3://usecase-etl-2-455865672536/output/daily_country_revenue.csv
Monthly output: s3://usecase-etl-2-455865672536/output/monthly_category_revenue.csv


## Step 1 — Extract the cleaned data

### Why this step is performed
In ETL terms, this is the **extract** phase: we load the cleaned source that is ready for downstream calculations and aggregation.


In [5]:
df = read_csv_aws_first(INPUT_S3_URI, LOCAL_INPUT_PATH)
print('Clean input shape:', df.shape)
display(df.head())


✅ Reading from S3: s3://usecase-etl-1-455865672536/processed/retail_cleaned.csv
Clean input shape: (489, 14)


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,InvoiceDateParsed,TransactionDate,Year,Month,Revenue,IsReturn
0,536365,71053,WHITE METAL LANTERN,6,2011-02-01 11:08:00,5.49,17889.0,Belgium,2011-02-01 11:08:00,2011-02-01,2011,2011-02,32.94,False
1,536365,21730,GLASS STAR FROSTED T-LIGHT HOLDER,2,2011-01-28 11:32:00,4.22,16943.0,Germany,2011-01-28 11:32:00,2011-01-28,2011,2011-01,8.44,False
2,536366,21730,GLASS STAR FROSTED T-LIGHT HOLDER,6,2011-02-05 08:48:00,5.80,18065.0,Netherlands,2011-02-05 08:48:00,2011-02-05,2011,2011-02,34.80,False
3,536366,22752,SET 7 BABUSHKA NESTING BOXES,4,2011-01-13 13:54:00,7.55,14512.0,United Kingdom,2011-01-13 13:54:00,2011-01-13,2011,2011-01,30.20,False
4,536366,21730,GLASS STAR FROSTED T-LIGHT HOLDER,6,2011-03-22 17:56:00,4.03,17075.0,Germany,2011-03-22 17:56:00,2011-03-22,2011,2011-03,24.18,False


## Step 2 — Transform for reporting

### Why this step is performed
Create fields that make aggregation easier.


In [6]:
df['TransactionDate'] = pd.to_datetime(df['TransactionDate'])
df['Category'] = df['Description'].fillna('UNKNOWN_ITEM').str.split().str[0]
df['Revenue'] = df['Quantity'] * df['UnitPrice']

display(df[['TransactionDate', 'Country', 'Category', 'Revenue']].head())


,TransactionDate,Country,Category,Revenue
0,2011-02-01,Belgium,WHITE,32.94
1,2011-01-28,Germany,GLASS,8.44
2,2011-02-05,Netherlands,GLASS,34.80
3,2011-01-13,United Kingdom,SET,30.20
4,2011-03-22,Germany,GLASS,24.18


## Step 3 — Aggregate into reporting outputs

### Why this step is performed
This is where row-level data becomes business-ready output.


In [7]:
daily_country_revenue = (
    df.groupby([df['TransactionDate'].dt.date.astype(str), 'Country'], as_index=False)['Revenue']
      .sum()
      .rename(columns={'TransactionDate': 'Date'})
)

monthly_category_revenue = (
    df.groupby(['Month', 'Category'], as_index=False)['Revenue']
      .sum()
)

display(daily_country_revenue.head())
display(monthly_category_revenue.head())


/tmp/ipykernel_1916/2479616822.py:3: FutureWarning: A grouping was used that is not in the columns of the DataFrame and so was excluded from the result. This grouping will be included in a future version of pandas. Add the grouping as a column of the DataFrame to silence this warning.
  .sum()


,Country,Revenue
0,France,7.74
1,United Kingdom,63.96
2,Belgium,20.75
3,Germany,50.78
4,Belgium,11.62


,Month,Category,Revenue
0,2011-01,ASSORTED,333.42
1,2011-01,CREAM,314.82
2,2011-01,GLASS,432.14
3,2011-01,HAND,1161.19
4,2011-01,KNITTED,558.09


## Step 4 — Load the ETL outputs back to S3

### Why this step is performed
The daily and monthly views are written back to S3 so they can be verified in the AWS console.


In [8]:
USE_LOCAL_FALLBACK = False

write_csv_aws_first(daily_country_revenue, OUT_DAILY_S3_URI, LOCAL_DAILY_PATH)
write_csv_aws_first(monthly_category_revenue, OUT_MONTHLY_S3_URI, LOCAL_MONTHLY_PATH)

print('Daily revenue output saved to:')
print(OUT_DAILY_S3_URI if not USE_LOCAL_FALLBACK else LOCAL_DAILY_PATH.resolve())
print('Monthly revenue output saved to:')
print(OUT_MONTHLY_S3_URI if not USE_LOCAL_FALLBACK else LOCAL_MONTHLY_PATH.resolve())

✅ Written to S3: s3://usecase-etl-2-455865672536/output/daily_country_revenue.csv
✅ Written to S3: s3://usecase-etl-2-455865672536/output/monthly_category_revenue.csv
Daily revenue output saved to:
s3://usecase-etl-2-455865672536/output/daily_country_revenue.csv
Monthly revenue output saved to:
s3://usecase-etl-2-455865672536/output/monthly_category_revenue.csv


In [9]:
import datetime, pytz;
print("Current Time in IST:", datetime.datetime.now(pytz.utc).astimezone(pytz.timezone('Asia/Kolkata')).strftime('%Y-%m-%d %H:%M:%S'))

Current Time in IST: 2026-09-14 09:03:22
